In [5]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
import os
from psycopg import Connection
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model= "llama-3.1-8b-instant")

In [4]:
MAX_TOKENS = 150

In [6]:
def call_model(state: MessagesState):
    messages = trim_messages(
        state['messages'],
        strategy= 'last',
        token_counter= count_tokens_approximately,
        max_tokens= MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages= messages))

    for message in messages:
        print(message.content)

    response = llm.invoke(messages)

    return {"messages": [response]}

In [7]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [8]:
DB_URI = os.getenv("DATABASE_URL")

In [13]:
conn = Connection.connect(
    DB_URI,
    autocommit= True,
    prepare_threshold= None 
)

In [14]:
checkpointer = PostgresSaver(conn)
checkpointer.setup()

In [15]:
graph = builder.compile(checkpointer=checkpointer)

# Thread 1 (remembers)
t1 = {"configurable": {"thread_id": "thread-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Aryan"}]}, t1)
out1 = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, t1)
print("Thread-1:", out1["messages"][-1].content)

Current Token Count -> 64
Hi, my name is Aryan
Nice to meet you, Aryan. Is there something I can help you with or would you like to chat?
What is my name?
Your name is Aryan.
Hi, my name is Aryan
Current Token Count -> 97
Hi, my name is Aryan
Nice to meet you, Aryan. Is there something I can help you with or would you like to chat?
What is my name?
Your name is Aryan.
Hi, my name is Aryan
Hi Aryan, it's nice to confirm your name again. How's your day going so far?
What is my name?
Thread-1: Your name is Aryan.
